In [ ]:
# Crear una lista con los nombres de los paquetes necesarios para el proyecto
paquetes <- c(
  "readxl",    # Permite leer archivos Excel (.xls y .xlsx)
  "readr",     # Facilita la lectura y escritura de archivos de texto y CSV
  "writexl",   # Permite exportar datos a archivos Excel (.xlsx)
  "forecast",  # Proporciona herramientas para modelado y pronóstico (ARIMA, ETS, etc.)
  "tseries",   # Incluye funciones para analisis de series temporales
  "astsa",     # Contiene funciones adicionales para analisis de series temporales (ej. sarima.for)
  "reticulate",# Permite integrar y ejecutar código Python desde R (opcional)
  "here"       # Facilita el manejo de rutas relativas dentro del proyecto
)

# Verifica que paquetes de la lista no estan instalados actualmente en el sistema
faltan <- setdiff(paquetes, rownames(installed.packages()))

# Si hay paquetes faltantes, los instala junto con sus dependencias
if (length(faltan)) install.packages(faltan, dependencies = TRUE)

In [ ]:
# Carga las librerias necesarias para el proyecto en la sesion actual de R

library(readxl)     # Para leer archivos Excel
library(readr)      # Para leer y escribir datos en formato texto/CSV
library(writexl)    # Para exportar datos a archivos Excel
library(forecast)   # Para modelar y pronosticar series temporales
library(tseries)    # Para analisis de series temporales
library(astsa)      # Para funciones adicionales de analisis de series temporales
library(here)       # Para manejar rutas relativas de forma segura

In [ ]:
# Elimina la carpeta local del proyecto si ya existe, para evitar conflictos con versiones anteriores
system("rm -rf portafolio-analisis-de-datos")

# Clona desde GitHub la ultima version del repositorio indicado
system("git clone https://github.com/vzoccola/portafolio-analisis-de-datos.git")

# Cambia el directorio de trabajo actual a la carpeta del proyecto recien clonada
setwd("portafolio-analisis-de-datos")

# Verifica la ubicación actual del directorio de trabajo
# Deberia mostrar: /content/portafolio-analisis-de-datos
getwd()

In [ ]:
# Define la ruta al archivo Excel, uniendo las carpetas y el nombre del archivo
# Tasas de mortalidad para hombres y mujeres de Chile, desde año 1992 a 2019
ruta_excel <- file.path("projects", "deuda-publica-pensiones-chile", "data", "Deaths Rates.xlsx")

# Lee tasas de mortalidad desde la hoja "CL_H" (hombres) del archivo Excel
hm.cl <- read_excel(ruta_excel, sheet = "CL_H")

# Lee tasas de mortalidad desde la hoja "CL_M" (mujeres) del archivo Excel
ml.cl <- read_excel(ruta_excel, sheet = "CL_M")

# Muestra las primeras filas del conjunto de datos de hombres
head(hm.cl)

# Muestra las primeras filas del conjunto de datos de mujeres
head(ml.cl)

In [ ]:
# Filtra las columnas correspondientes a los años 1992–2000 (posiciones 1 a 9) para hombres
hm.cl.00 <- hm.cl[, c(1:9)]

# Filtra las columnas correspondientes a los años 1992–2000 (posiciones 1 a 9) para mujeres
ml.cl.00 <- ml.cl[, c(1:9)]

# Reemplaza en los datos de hombres las tasas mayores a 1 por 1 (corrige posibles errores)
hm.cl.00 <- replace(hm.cl.00, hm.cl.00 > 1, 1)

# Reemplaza en los datos de hombres las tasas iguales a 0 por 1 (evita problemas al calcular logaritmos)
hm.cl.00 <- replace(hm.cl.00, hm.cl.00 == 0, 1)

# Reemplaza en los datos de mujeres las tasas mayores a 1 por 1
ml.cl.00 <- replace(ml.cl.00, ml.cl.00 > 1, 1)

# Reemplaza en los datos de mujeres las tasas iguales a 0 por 1
ml.cl.00 <- replace(ml.cl.00, ml.cl.00 == 0, 1)

In [ ]:
# Define funcion estim1 para estimar los parametros del modelo Lee–Carter
# Entrada:
#   set: matriz de tasas (edad × tiempo) con dimensiones 111 × 51
# Salida (lista):
#   a_x: nivel promedio por edad
#   b_x: sensibilidad por edad al factor temporal
#   k_t: factor temporal comun
#   prop_var: proporción de varianza explicada por el primer componente
estim1 = function(set)
{
  T <- dim(set)[2]
  log(set) -> lml
  rowMeans(lml) -> a.x
  lml - matrix(rowMeans(lml), 111, 51) -> lml0
  svd(lml0, T, T) -> svdM
  (svd(lml0, T, T)$d) -> sv
  svdM$u[,1] / sum(svdM$u[,1]) -> b.x
  (svdM$v)[,1] * svdM$d[1] * sum(svdM$u[,1]) -> k.t
  list(a.x, b.x, k.t, (sv[1]^2) / sum(sv^2))
}

In [ ]:
# Estima los parametros del modelo Lee–Carter para hombres usando los datos filtrados
estim1(hm.cl.00)[[1]] -> ax.h.cl  # Nivel promedio por edad (a_x) para hombres
estim1(hm.cl.00)[[2]] -> bx.h.cl  # Sensibilidad por edad (b_x) para hombres
estim1(hm.cl.00)[[3]] -> kt.h.cl  # Factor temporal (k_t) para hombres

# Estima los parametros del modelo Lee–Carter para mujeres usando los datos filtrados
estim1(ml.cl.00)[[1]] -> ax.m.cl  # Nivel promedio por edad (a_x) para mujeres
estim1(ml.cl.00)[[2]] -> bx.m.cl  # Sensibilidad por edad (b_x) para mujeres
estim1(ml.cl.00)[[3]] -> kt.m.cl  # Factor temporal (k_t) para mujeres

In [ ]:
# Configura parametros graficos: una sola ventana de graficos (1x1) y margenes personalizados
par(mfrow = c(1,1), mar = c(4,4,2,1))

# Grafica la serie temporal k_t para hombres (1992–2000)
plot(1:length(kt.h.cl), kt.h.cl,
     type = "b",              # Lineas y puntos
     pch = 16,                # Simbolo para puntos (círculo relleno)
     lwd = 2,                 # Grosor de linea
     col = "blue",            # Color azul para hombres
     ylim = range(c(kt.h.cl, kt.m.cl)), # Escala vertical comun para hombres y mujeres
     xlab = "Año índice (1992–2000)",   # Etiqueta del eje X
     ylab = expression(k[t]))           # Etiqueta del eje Y con notacion matematica

# Agrega la serie k_t para mujeres sobre el mismo grafico
points(1:length(kt.m.cl), kt.m.cl,
       type = "b",            # Lineas y puntos
       pch = 15,              # Simbolo para puntos (cuadrado relleno)
       lwd = 2,               # Grosor de linea
       col = "darkgreen")     # Color verde oscuro para mujeres

# Agrega leyenda en la esquina superior derecha
legend("topright",
       legend = c("Hombres", "Mujeres"), # Etiquetas
       pch = c(16,15),                   # Simbolos correspondientes
       col = c("blue","darkgreen"),      # Colores correspondientes
       bty = "n")                        # Sin borde de caja

In [ ]:
# Configura la ventana grafica para mostrar 4 graficos (2 filas × 2 columnas) con margenes personalizados
par(mfrow = c(2,2), mar = c(4,4,2,1))

# Grafica la primera diferencia de k_t para hombres
plot(diff(kt.h.cl), type = "b", pch = 16, lwd = 2, col = "blue",
     main = expression(Delta*k[t]~"(Hombres)"), # Titulo con notacion matematica
     ylab = "", xlab = "")                      # Sin etiquetas de ejes

# Grafica la primera diferencia de k_t para mujeres
plot(diff(kt.m.cl), type = "b", pch = 15, lwd = 2, col = "darkgreen",
     main = expression(Delta*k[t]~"(Mujeres)"), # Titulo con notacion matematica
     ylab = "", xlab = "")                      # Sin etiquetas de ejes

# Muestra la funcion de autocorrelacion (ACF) de la primera diferencia de k_t para hombres
acf(diff(kt.h.cl), main = "ACF Δk_t Hombres")

# Muestra la funcion de autocorrelacion (ACF) de la primera diferencia de k_t para mujeres
acf(diff(kt.m.cl), main = "ACF Δk_t Mujeres")

# Restablece la configuracion de la ventana grafica a una sola figura
par(mfrow = c(1,1))

In [ ]:
# Ajusta un modelo ARIMA automaticamente para la serie k_t de hombres
fit.h <- auto.arima(kt.h.cl)   # Devuelve el modelo ARIMA optimo segun criterios estadisticos

# Ajusta un modelo ARIMA automaticamente para la serie k_t de mujeres
fit.m <- auto.arima(kt.m.cl)   # Devuelve el modelo ARIMA optimo segun criterios estadisticos

# Muestra el resumen del modelo ARIMA ajustado para hombres
fit.h

# Muestra el resumen del modelo ARIMA ajustado para mujeres
fit.m

In [ ]:
# Define el numero de años a proyectar (2001–2050)
n.ahead <- 50

# Proyección de k_t para hombres usando un modelo ARIMA(0,1,0) sin estacionalidad
# S=0 indica que no se considera componente estacional
moh.cl <- sarima.for(kt.h.cl, n.ahead = n.ahead,
                     p = 0, d = 1, q = 0,   # Parametros del componente no estacional
                     P = 0, D = 0, Q = 0,   # Parametros del componente estacional
                     S = 0,                 # Periodo estacional inexistente
                     plot = FALSE)          # No mostrar grafico automatico

# Proyeccion de k_t para mujeres usando el mismo modelo ARIMA(0,1,0) sin estacionalidad
mom.cl <- sarima.for(kt.m.cl, n.ahead = n.ahead,
                     p = 0, d = 1, q = 0,
                     P = 0, D = 0, Q = 0,
                     S = 0,
                     plot = FALSE)

In [ ]:
# Calcula la matriz de predicciones para mujeres (111 edades × 50 años proyectados)
# Formula del modelo Lee–Carter: m_x,t = exp(a_x + b_x * k_t) × 100000 (tasa por 100.000 habitantes)
pred.muj <- exp(
  ax.m.cl +                                      # Nivel promedio por edad
  bx.m.cl * matrix(mom.cl$pred,                  # Factor temporal proyectado
                   nrow = 111,                   # Numero de edades
                   ncol = n.ahead,               # Numero de años proyectados
                   byrow = TRUE)                 # Repetir k_t por filas
) * 100000                                          # Escala de tasas por 100,000

# Calcula la matriz de predicciones para hombres (111 edades × 50 años proyectados)
pred.hom <- exp(
  ax.h.cl +                                      # Nivel promedio por edad
  bx.h.cl * matrix(moh.cl$pred,                  # Factor temporal proyectado
                   nrow = 111,
                   ncol = n.ahead,
                   byrow = TRUE)
) * 100000

In [ ]:
# Define las columnas correspondientes en los datos originales y en las proyecciones
k_obs_2010 <- 19   # Columna en el Excel para el año 2010 (observado)
k_pred_2010 <- 10  # Columna en las proyecciones (2000 + 10 = 2010)

# Configura la ventana grafica para mostrar dos graficos lado a lado
par(mfrow = c(1,2), mar = c(4,4,2,1))

# --- Grafico 1: Mujeres año 2010 ---
plot(1:111, log(unlist(ml.cl[, k_obs_2010])),  # Tasas observadas (log) para 2010
     type = "l", lwd = 2, col = "black",       # Linea negra continua
     xlab = "Edad", ylab = "log Tasa de muerte",
     main = "Mujeres • 2010", ylim = c(-9,1))  # Escala Y ajustada
lines(1:111, log(pred.muj[, k_pred_2010] / 1e5), # Tasas proyectadas (log) para 2010
      lwd = 2, col = "blue", lty = 3)           # Linea azul punteada
legend("topright", legend = c("Observado", "Proyectado"),
       lty = c(1,3), col = c("black","blue"), bty = "n")

# --- Grafico 2: Mujeres año 2020 ---
k_obs_2020 <- 29   # Columna en el Excel para 2020 (observado)
k_pred_2020 <- 20  # Columna en las proyecciones (2000 + 20 = 2020)

plot(1:111, log(unlist(ml.cl[, k_obs_2020])),  # Tasas observadas (log) para 2020
     type = "l", lwd = 2, col = "gray30",      # Linea gris oscuro
     xlab = "Edad", ylab = "log Tasa de muerte",
     main = "Mujeres • 2020")
lines(1:111, log(pred.muj[, k_pred_2020] / 1e5), # Tasas proyectadas (log) para 2020
      lwd = 2, col = "red", lty = 3)             # Linea roja punteada
legend("bottomleft", legend = c("Observado", "Proyectado"),
       lty = c(1,3), col = c("gray30","red"), bty = "n")

# Restablece la configuracion grafica a una sola figura
par(mfrow = c(1,1))

In [ ]:
# Lee la hoja "CL_H_S" del archivo Excel, que contiene datos para hombres
hm.cl.s <- readxl::read_excel(ruta_excel, sheet = "CL_H_S")

# Reemplaza valores extremos en toda la hoja:
#   - Si la tasa es mayor a 1, se reemplaza por 1
#   - Si la tasa es igual a 0, se reemplaza por 1
# Esto evita problemas al aplicar logaritmos y corrige valores atipicos
hm.cl.s[] <- lapply(hm.cl.s, \(x) {
  x[x > 1 | x == 0] <- 1
  x
})

# Muestra las primeras filas del dataset corregido
head(hm.cl.s)

# Muestra las dimensiones del dataset (número de filas y columnas)
dim(hm.cl.s)

In [ ]:
# estim1()  —  version para tamaño (111 × 9)

estim1 <- function(set)
{
  T <- dim(set)[2]
  log(set) -> lml
  rowMeans(lml) -> a.x
  lml - matrix(rowMeans(lml), 111, 9) -> lml0
  svd(lml0, T, T) -> svdM
  (svd(lml0, T, T)$d) -> sv
  (sv[1]^2) / sum(sv^2)
  svdM$u[,1] / sum(svdM$u[,1]) -> b.x
  (svdM$v)[,1] * svdM$d[1] * sum(svdM$u[,1]) -> k.t
  list(a.x, b.x, k.t, (sv[1]^2) / sum(sv^2))
}

In [ ]:
# Convierte el dataset a matriz eliminando la columna "Edad"
hm.mat <- as.matrix(hm.cl.s[ , !names(hm.cl.s) %in% "Edad"])

# Crea una base vacia para almacenar las proyecciones finales
# 111 filas (edades) y 0 columnas (inicialmente vacia)
BD.hm.cl <- matrix(nrow = 111, ncol = 0)

# Bucle de ventana movil (20 ventanas de 9 años cada una)
for (i in 1:20) {

  # Extrae la ventana de 9 años consecutivos
  # Ej.: 1992–2000, 1993–2001, etc.
  hm.cl.00 <- hm.mat[ , i:(i + 8), drop = FALSE]

  # Estima parametros Lee–Carter para la ventana
  ax.h <- estim1(hm.cl.00)[[1]]  # Nivel promedio por edad
  bx.h <- estim1(hm.cl.00)[[2]]  # Sensibilidad por edad
  kt.h <- estim1(hm.cl.00)[[3]]  # Factor temporal

  # Proyecta k_t 50 años hacia adelante con ARIMA(0,1,0) sin estacionalidad
  moh <- sarima.for(
           kt.h,
           n.ahead = 50,
           p = 0, d = 1, q = 0,     # Componente no estacional
           P = 0, D = 0, Q = 0,     # Componente estacional
           S = 0,                   # Sin estacionalidad
           no.constant = FALSE,
           plot = FALSE)

  # Crea la matriz de tasas proyectadas (111 edades × 50 años)
  predmr <- matrix(0, nrow = 0, ncol = 50)
  for (l in 1:111) {
    # Aplica formula Lee–Carter y escala a tasas por 100.000 habitantes
    fila <- t(exp(ax.h[l] + bx.h[l] * moh$pred)) * 100000
    predmr <- rbind(predmr, fila)
  }

  # Agrega el bloque proyectado a la base general
  # Dividido por 100.000 para volver a la escala de tasas 0–1
  BD.hm.cl <- cbind(BD.hm.cl, predmr / 100000)
}

# Comprobaciones rapidas
dim(BD.hm.cl)            # Dimensiones de la matriz final (→ 111 × 1000)
summary(BD.hm.cl[ , 1:5])# Resumen estadistico de las primeras 5 columnas
head(BD.hm.cl)           # Primeras filas de la matriz
dim(BD.hm.cl)            # Confirmacion de dimensiones

In [ ]:
# Post-procesamiento: asegura que las tasas no superen 1 (proporciones en escala 0–1)
BD.hm.cl[BD.hm.cl > 1] <- 1

# Exporta la matriz a Excel multiplicando por 100.000 para devolverla a escala de tasas por 100.000 habitantes
# El archivo se guardara en el directorio actual de trabajo (en Colab: /content/)
writexl::write_xlsx(as.data.frame(BD.hm.cl * 100000), "BD.hm.cl.xlsx")

# Mensaje de confirmacion en consola
cat("Archivo 'BD.hm.cl.xlsx' escrito en el directorio de trabajo\n")

In [ ]:
# Calculo del Costo Neto Unitario (CNU) para edades desde 65 años (fila 67) hasta 110+
# Usa tasas de interes anuales del periodo 2000–2019 para el descuento

# Vector con tasas de interés (rho) para cada una de las 20 ventanas de proyección
rho <- c(0.0519, 0.0546, 0.0427, 0.0415, 0.0340,
         0.0347, 0.0325, 0.0328, 0.0390, 0.0367,
         0.0308, 0.0318, 0.0321, 0.0285, 0.0227,
         0.0276, 0.0238, 0.0274, 0.0280, 0.0139)

# Inicializa estructuras para almacenar resultados
matrix.cnu.hm.cl <- c()      # Matriz final (cada fila = edad, cada columna = ventana)
cnu.hm.cl <- rep(0, 20)      # Vector auxiliar para CNU en una edad especifica

# Bucle principal por edad (67 ≈ edad 65 en la matriz) hasta 111 (110+)
for (i in 67:111) {

  # Bucle por ventana de proyeccion (20 ventanas)
  for (k in 1:20) {
    r <- rho[k]              # Tasa de interes de la ventana k
    t <- 1 / (1 + r)         # Factor de descuento
    multiplicaciones <- c()  # Probabilidades acumuladas de sobrevivir
    m <- 1                   # Probabilidad inicial de supervivencia

    # Calculo de probabilidades acumuladas de supervivencia para la edad i-1 hacia adelante
    for (d in 1:(111 - i + 1)) {
      # Probabilidad de sobrevivir un año mas: 1 - tasa de mortalidad proyectada
      m <- m * (1 - BD.hm.cl[i - 1 + d, d + (k - 1) * 50])
      multiplicaciones <- cbind(multiplicaciones, m)
    }

    # Calculo de g: valor presente esperado de pagos, incluyendo t = 0
    g <- 1
    for (f in 1:(111 - i + 1)) {
      g <- g + multiplicaciones[f] * (t^f)
    }

    # Almacena el CNU para la ventana k
    cnu.hm.cl[k] <- g
  }

  # Añade la fila de CNU para la edad i al resultado final
  matrix.cnu.hm.cl <- rbind(matrix.cnu.hm.cl, t(cnu.hm.cl))
}

# Muestra los primeros resultados de CNU (edades 65–70)
head(matrix.cnu.hm.cl)

# Muestra las dimensiones de la matriz CNU
dim(matrix.cnu.hm.cl)

In [ ]:
# Configura márgenes para el gráfico
par(mar = c(4,4,2,1))

# Grafica el CNU para la edad 75 años (fila 11 de la matriz: 65 + 10 años)
plot(2000:2019, matrix.cnu.hm.cl[11, ],
     type = "l", lwd = 2, col = "green",
     ylim = c(6, 18),                    # Rango en eje Y
     xlab = "Año", ylab = "CNU",
     main = "CNU • Hombres (Chile)")

# Agrega linea para la edad 65 años (fila 1 de la matriz)
lines(2000:2019, matrix.cnu.hm.cl[1, ],
      lwd = 2, col = "black")

# Agrega linea para la edad 70 años (fila 6 de la matriz)
lines(2000:2019, matrix.cnu.hm.cl[6, ],
      lwd = 2, col = "red")

# Agrega leyenda
legend("top",
       legend = c("CNU 65 años", "CNU 70 años", "CNU 75 años"),
       col    = c("black", "red", "green"),
       lty    = 1,
       lwd    = 2,
       bty    = "n")

In [ ]:
# Lectura de porcentajes de poblacion para hombres desde distintos años base
# Se elimina la primera columna (edad) dejando solo porcentajes por año
porcentaje_poblacion00_cl_h <- read_excel(
  file.path("projects", "deuda-publica-pensiones-chile", "data", "% de la Población-00.xlsx"),
  sheet = "CL_H")[,-1]
porcentaje_poblacion04_cl_h <- read_excel(
  file.path("projects", "deuda-publica-pensiones-chile", "data", "% de la Poblacion-04.xlsx"),
  sheet = "CL_H")[,-1]
porcentaje_poblacion10_cl_h <- read_excel(
  file.path("projects", "deuda-publica-pensiones-chile", "data", "% de la Poblacion-10.xlsx"),
  sheet = "CL_H")[,-1]
porcentaje_poblacion15_cl_h <- read_excel(
  file.path("projects", "deuda-publica-pensiones-chile", "data", "% de la Población-15.xlsx"),
  sheet = "CL_H")[,-1]

# Lectura de tasas de crecimiento de la poblacion de 65+ años para hombres
creciemiento_cl_h <- read_excel(
  file.path("projects","deuda-publica-pensiones-chile","data","Tasas de Crecimiento (F).xlsx"),
  sheet = "CL_H")

# PIB de Chile (millones USD) para años 2000–2019, serie manual
pib_cl <- c(3442.99, 3298.37, 3178.73, 3318.13, 3808.60, 4513.42, 4996.72,
            5332.28, 5943.26, 6076.57, 7353.36, 8074.36, 8319.85, 8275.59,
            7623.91, 7029.97, 7032.71, 7796.25, 8203.33, 7825.44)

# Lectura de PIB desde archivo Excel
pib <- read_excel(
  file.path("projects","deuda-publica-pensiones-chile","data","Gasto Pensión PIB.xlsx"),
  sheet = "PIB_Nom")

# Tasas de interes a 10 años (2000–2019)
rho_cl <- c(0.0519, 0.0546, 0.0427, 0.0415, 0.0340, 0.0347, 0.0325, 0.0328, 0.0390, 0.0367,
            0.0308, 0.0318, 0.0321, 0.0285, 0.0227, 0.0276, 0.0238, 0.0274, 0.0280, 0.0139)

# Lectura de CNU proyectados para hombres, entregados por el docente
cnu.hm.cl_p <- read_excel(
  file.path("projects","deuda-publica-pensiones-chile","data","cnu.hm.cl.xlsx"))

# Inicializacion de vectores para valores presentes:
# VP1.cl.h = valor presente en el momento actual (año j)
# VP2.cl.h = valor presente de pagos futuros (desde t+1 en adelante)
VP1.cl.h <- rep(0, 20)
VP2.cl.h <- rep(0, 20)

In [ ]:
# Calculo de Valor Presente (VP) por quinquenio para hombres

# ---- Periodo 2000–2004 ----
for (j in 1:5) {  # Año dentro del quinquenio
  for (i in 1:36) {  # Edad desde 65 a 100+
    VP <- porcentaje_poblacion00_cl_h[j, i] *
          cnu.hm.cl_p[i, (j - 1) * 50 + j]   # CNU correspondiente
    VP1.cl.h[j] <- VP1.cl.h[j] + VP * pib_cl[j]  # Multiplica por PIB del año
  }
}
# VP para pagos futuros desde t+1
for (k in 1:5) {  # Ventana
  for (j in 1:50) {  # Año futuro proyectado
    for (i in 1:36) {
      VP <- ((creciemiento_cl_h[j, 2] + 1) *
             porcentaje_poblacion00_cl_h[j + 1, i] *
             cnu.hm.cl_p[i, j + 50 * (k - 1)]) / ((1 + rho_cl[k])^j)
      VP2.cl.h[k] <- VP2.cl.h[k] + VP * pib_cl[k]
    }
  }
}

# ---- Periodo 2005–2009 ----
for (j in 6:10) {
  for (i in 1:36) {
    VP <- porcentaje_poblacion04_cl_h[j - 5, i] *
          cnu.hm.cl_p[i, 250 + j + 50 * (j - 6)]
    VP1.cl.h[j] <- VP1.cl.h[j] + VP * pib_cl[j]
  }
}
for (k in 6:10) {
  for (j in 1:45) {
    for (i in 1:36) {
      VP <- ((creciemiento_cl_h[j, 3] + 1) *
             porcentaje_poblacion04_cl_h[j + 1, i] *
             cnu.hm.cl_p[i, k + 250 + 50 * (k - 6)]) / ((1 + rho_cl[k])^j)
      VP2.cl.h[k] <- VP2.cl.h[k] + VP * pib_cl[k]
    }
  }
}

# ---- Periodo 2010–2014 ----
for (j in 11:15) {
  for (i in 1:36) {
    VP <- porcentaje_poblacion10_cl_h[j - 10, i] *
          cnu.hm.cl_p[i, 500 + j + 50 * (j - 11)]
    VP1.cl.h[j] <- VP1.cl.h[j] + VP * pib_cl[j]
  }
}
for (k in 11:15) {
  for (j in 1:40) {
    for (i in 1:36) {
      VP <- ((creciemiento_cl_h[j, 4] + 1) *
             porcentaje_poblacion10_cl_h[j + 1, i] *
             cnu.hm.cl_p[i, k + 500 + 50 * (k - 11)]) / ((1 + rho_cl[k])^j)
      VP2.cl.h[k] <- VP2.cl.h[k] + VP * pib_cl[k]
    }
  }
}

# ---- Periodo 2015–2019 ----
for (j in 16:20) {
  for (i in 1:36) {
    VP <- porcentaje_poblacion15_cl_h[j - 15, i] *
          cnu.hm.cl_p[i, 750 + j + 50 * (j - 16)]
    VP1.cl.h[j] <- VP1.cl.h[j] + VP * pib_cl[j]
  }
}
for (k in 16:20) {
  for (j in 1:35) {
    for (i in 1:36) {
      VP <- ((creciemiento_cl_h[j, 5] + 1) *
             porcentaje_poblacion15_cl_h[j + 1, i] *
             cnu.hm.cl_p[i, k + 750 + 50 * (k - 15)]) / ((1 + rho_cl[k])^j)
      VP2.cl.h[k] <- VP2.cl.h[k] + VP * pib_cl[k]
    }
  }
}

# Calculo final de deuda previsional y relación con PIB

# Suma de VP actual + VP de pagos futuros
deuda.cl.h <- as.numeric(VP1.cl.h) + as.numeric(VP2.cl.h)

# Obtiene PIB correspondiente a los años 2000–2019 desde archivo Excel
pib.cl <- pib[1:20, 7]

# Calcula deuda como porcentaje del PIB
deuda_pib.cl.h <- deuda.cl.h / pib.cl

In [ ]:
# Grafica la deuda previsional como porcentaje del PIB para hombres en Chile (2000–2019)
plot(2000:2019, t(deuda_pib.cl.h),
     type = "l",              # Linea continua
     ylab = "% PIB",          # Etiqueta del eje Y
     xlab = "Año",            # Etiqueta del eje X
     ylim = c(4, 12),         # Limite del eje Y
     main = "Deuda como % PIB Hombres • Chile")